<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:28px 32px;margin-bottom:8px;box-sizing:border-box;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#4C8DFF;margin-bottom:10px;">FRANCE DATA MARKET &middot; EXPLORATION</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:28px;color:#F2F3F5;letter-spacing:-0.02em;margin-bottom:8px;">State of the data job market in France</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:14px;color:#9BA1AC;">Manual exploration notebook, run against the current warehouse.duckdb. Structure mirrors dashboard/requetes.sql: same grain, same definitions, so figures found here and in the generated report never diverge.</div></div>

In [1]:
import duckdb
import pandas as pd
import plotly.io as pio

import sys
sys.path.insert(0, '../dashboard')
from theme import BLUE, AMBER  # noqa: F401  -- registers the 'dark' Plotly template

con = duckdb.connect('../data/warehouse.duckdb', read_only=True)
pio.templates.default = 'dark'

<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">00 &middot; SCOPE</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Corpus size and top-line KPIs</div></div>

In [2]:
scope = con.execute('''
    select count(*) as offers,
           count(case when is_canonical_listing then 1 end) as listings
    from fct_job_offer
''').df()
scope

,offers,listings
0,1132,981


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">01 &middot; MARKET FLOW</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">What appears, what disappears</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Measured on fct_weekly_market_flow (actual presence per pull), never on the accumulated fct_weekly_market corpus -- see the README for why.</div></div>

In [3]:
flow = con.execute('''
    select week_start_date, weeks_since_previous, active_offer_count,
           new_offer_count, exit_count, exit_rate_pct
    from fct_weekly_market_flow
    order by week_start_date
''').df()
flow

,week_start_date,weeks_since_previous,active_offer_count,new_offer_count,exit_count,exit_rate_pct
0,2026-07-13,<NA>,552,552,<NA>,NaN
1,2026-08-24,6,497,408,463,83.9
2,2026-08-31,1,654,179,25,5.0


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">02 &middot; COMPENSATION</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Median advertised salary</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Filtered on annual_salary_plausible, grouped by canonical listing.</div></div>

In [4]:
salary_by_category = con.execute('''
    select employer_category, count(*) as n, median(salary_min) as median_salary
    from fct_job_offer
    where salary_period = 'annual' and annual_salary_plausible
      and is_canonical_listing
    group by employer_category
    order by median_salary desc
''').df()
salary_by_category

,employer_category,n,median_salary
0,INTERMEDIARY_RECLASSIFIED,3,65000.0
1,ANONYMOUS,24,45000.0
2,INTERMEDIARY,97,44000.0
3,DIRECT_EMPLOYER,134,42500.0


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">03 &middot; SALARY TRANSPARENCY</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Salary disclosure in job offers</div></div>

In [5]:
transparency = con.execute('''
    select employer_category,
           round(100.0 * count(distinct case when salary_mentioned then job_offer_id end)
                 / nullif(count(distinct job_offer_id), 0), 1) as rate_pct
    from fct_job_offer
    where is_canonical_listing
    group by employer_category
    order by rate_pct desc
''').df()
transparency

,employer_category,rate_pct
0,INTERMEDIARY,54.6
1,DIRECT_EMPLOYER,37.8
2,INTERMEDIARY_RECLASSIFIED,12.5
3,ANONYMOUS,9.1


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">04 &middot; GEOGRAPHY</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Top communes by offer count</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Counted by offer, not by listing -- a position opened in several communes represents an opportunity in each.</div></div>

In [6]:
geo = con.execute('''
    select c.commune_name, count(distinct o.job_offer_id) as offer_count
    from fct_job_offer o
    join dim_commune c on c.commune_key = o.commune_key
    where c.commune_name is not null and c.commune_name != 'UNRESOLVED'
    group by c.commune_name
    order by offer_count desc
    limit 10
''').df()
geo

,commune_name,offer_count
0,Paris,165
1,Lyon,52
2,Nantes,32
3,Nanterre,32
4,Courbevoie,31
5,Toulouse,29
6,Lille,25
7,Bordeaux,16
8,Boulogne-Billancourt,14
9,Puteaux,14


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">05 &middot; TECHNOLOGIES</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Most requested technologies</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Empty if the local extraction dump isn't available (CI_WITHOUT_EXTRACTION).</div></div>

In [7]:
skills = con.execute('''
    select t.technology, count(distinct t.job_offer_id) as offer_count
    from fct_job_offer_technology t
    join fct_job_offer o using (job_offer_id)
    where o.is_canonical_listing
    group by t.technology
    order by offer_count desc
    limit 10
''').df()
skills

,technology,offer_count
0,Python,298
1,SQL,271
2,Power BI,191
3,Databricks,66
4,AWS,66
5,Azure,63
6,Snowflake,57
7,Tableau,56
8,Spark,53
9,Git,52


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">06 &middot; DOMAINS</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Domains of intervention</div></div>

In [8]:
domains = con.execute('''
    select d.normalized_domain, count(distinct d.job_offer_id) as offer_count
    from fct_job_offer_domain d
    join fct_job_offer o using (job_offer_id)
    where o.is_canonical_listing
      and d.normalized_domain in (select distinct canonical_domain from mapping_domaines)
    group by d.normalized_domain
    order by offer_count desc
''').df()
domains

,normalized_domain,offer_count
0,Data Analysis,196
1,Data Governance,181
2,Business Intelligence,140
3,Machine Learning,112
4,Data Engineering,104
5,Data Science,102
6,Data Quality,93
7,Project Management,89
8,Data Architecture,64
9,Cloud computing,56


In [9]:
import plotly.graph_objects as go
import numpy as np
from IPython.display import HTML
from theme import (horizontal_bar_chart, column_chart, grouped_bar_chart,
                    line_chart, BLUE, AMBER, PAPER, PAPER_MUTED,
                    SLATE, SLATE_RAISED, HAIRLINE, MONO_FONT)

<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">07 &middot; SQL vs PYTHON</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Deep dive: the bilingual market</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Canonical listings only. Technologies extracted by mistral-nemo (LLM).</div></div>

In [10]:
segments = con.execute('''
    with offer_techs as (
        select distinct o.job_offer_id,
            max(case when lower(t.technology) = 'sql' then 1 else 0 end) as has_sql,
            max(case when lower(t.technology) = 'python' then 1 else 0 end) as has_python
        from fct_job_offer o
        left join fct_job_offer_technology t on o.job_offer_id = t.job_offer_id
        where o.is_canonical_listing
        group by o.job_offer_id
    )
    select
        case
            when has_sql = 1 and has_python = 1 then 'Both SQL & Python'
            when has_sql = 1 then 'SQL only'
            when has_python = 1 then 'Python only'
            else 'Neither'
        end as profile,
        count(*) as n
    from offer_techs
    group by profile
    order by n desc
''').df()
total = segments['n'].sum()
segments['label'] = segments.apply(lambda r: f"{r['n']}  ({100*r['n']/total:.0f}%)", axis=1)
fig = horizontal_bar_chart(
    segments.to_dict('records'), 'profile', 'n',
    'SQL vs Python composition',
    labels=dict(zip(segments['profile'], segments['label'])))
fig

In [11]:
domain_sp = con.execute('''
    with offer_techs as (
        select distinct job_offer_id,
            max(case when lower(technology) = 'sql' then 1 else 0 end) as has_sql,
            max(case when lower(technology) = 'python' then 1 else 0 end) as has_python
        from fct_job_offer_technology
        group by job_offer_id
    )
    select d.normalized_domain,
           sum(ot.has_sql)    as sql_count,
           sum(ot.has_python) as python_count
    from fct_job_offer_domain d
    join fct_job_offer o on d.job_offer_id = o.job_offer_id
    join offer_techs ot on d.job_offer_id = ot.job_offer_id
    where o.is_canonical_listing
      and d.normalized_domain in (
          select distinct canonical_domain from mapping_domaines)
    group by d.normalized_domain
    order by (sum(ot.has_sql) + sum(ot.has_python)) desc
''').df()

# Horizontal grouped bars — Blue for SQL, Amber for Python
domains_sorted = domain_sp.sort_values('sql_count', ascending=True)
fig = go.Figure()
fig.add_trace(go.Bar(
    y=domains_sorted['normalized_domain'], x=domains_sorted['sql_count'],
    name='SQL', orientation='h', marker_color=BLUE,
    text=domains_sorted['sql_count'], textposition='outside',
    textfont=dict(family=MONO_FONT, size=10, color=PAPER)))
fig.add_trace(go.Bar(
    y=domains_sorted['normalized_domain'], x=domains_sorted['python_count'],
    name='Python', orientation='h', marker_color=AMBER,
    text=domains_sorted['python_count'], textposition='outside',
    textfont=dict(family=MONO_FONT, size=10, color=PAPER)))
fig.update_layout(template='dark', barmode='group',
    height=max(300, 42 * len(domains_sorted) + 60),
    xaxis=dict(showticklabels=False, showgrid=True, gridcolor=HAIRLINE),
    yaxis=dict(showgrid=False),
    margin=dict(l=160, r=60, t=16, b=16))
fig

In [12]:
sal_profile = con.execute('''
    with offer_techs as (
        select distinct job_offer_id,
            max(case when lower(technology) = 'sql' then 1 else 0 end) as has_sql,
            max(case when lower(technology) = 'python' then 1 else 0 end) as has_python
        from fct_job_offer_technology
        group by job_offer_id
    )
    select
        case
            when ot.has_sql = 1 and ot.has_python = 1 then 'Both'
            when ot.has_sql = 1 then 'SQL only'
            when ot.has_python = 1 then 'Python only'
            else 'Neither'
        end as profile,
        count(*) as n,
        median(o.salary_min) as median_salary
    from fct_job_offer o
    left join offer_techs ot on o.job_offer_id = ot.job_offer_id
    where o.salary_period = 'annual'
      and o.annual_salary_plausible
      and o.is_canonical_listing
    group by profile
    order by median_salary desc
''').df()
fig = column_chart(
    sal_profile.to_dict('records'), 'profile', 'median_salary',
    'Median salary by tech profile', suffix=' €',
    labels={r['profile']: f"{int(r['median_salary']):,} € (n={int(r['n'])})".replace(',', ' ')
            for _, r in sal_profile.iterrows()})
fig

In [13]:
companions = con.execute('''
    with both_offers as (
        select a.job_offer_id
        from fct_job_offer_technology a
        join fct_job_offer_technology b on a.job_offer_id = b.job_offer_id
        where lower(a.technology) = 'sql' and lower(b.technology) = 'python'
    )
    select t.technology, count(distinct t.job_offer_id) as offer_count
    from fct_job_offer_technology t
    join both_offers bo on t.job_offer_id = bo.job_offer_id
    where lower(t.technology) != 'sql' and lower(t.technology) != 'python'
    group by t.technology
    order by offer_count desc
    limit 15
''').df()
fig = horizontal_bar_chart(
    companions.to_dict('records'), 'technology', 'offer_count',
    'Top 15 companion techs (SQL + Python offers)')
fig

<div style="margin:24px 0;padding:16px 20px 16px 24px;border-left:3px solid #4C8DFF;background:#1E2128;border-radius:0 8px 8px 0; box-sizing:border-box;"><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:15px;color:#F2F3F5;font-weight:700;margin-bottom:6px;">&#128161; Bilingual is the norm</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:13px;color:#9BA1AC;line-height:1.7;">57% of offers requiring at least one of SQL or Python demand <b>both</b>. The 6 k€ salary premium for bilingual profiles (45 k€ vs 39 k€ for SQL-only) suggests employers reward breadth over depth in a single language. SQL dominates governance and quality roles; Python leads in ML, Data Science, and Data Engineering — the two languages partition the field more by <i>domain</i> than by seniority.</div></div>

<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">08 &middot; dbt</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">An emerging signal</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">dbt mentions from LLM extraction + full-text search on job descriptions.</div></div>

In [14]:
dbt_listings = int(con.execute(
    "select count(distinct o.job_offer_id) "
    "from fct_job_offer o "
    "join fct_job_offer_technology t on o.job_offer_id = t.job_offer_id "
    "where o.is_canonical_listing and lower(t.technology) = 'dbt'"
).df().iloc[0, 0])

total_listings = int(con.execute(
    "select count(*) from fct_job_offer where is_canonical_listing"
).df().iloc[0, 0])

dbt_text_mentions = int(con.execute(
    "select count(distinct job_offer_id) from fct_job_offer "
    "where is_canonical_listing and lower(job_description) like '%dbt%'"
).df().iloc[0, 0])

dbt_snow_pct = float(con.execute(
    "with dbt_all as ("
    "  select distinct job_offer_id from fct_job_offer_technology"
    "  where lower(technology) = 'dbt')"
    " select round(100.0 * count(distinct t.job_offer_id)"
    "  / (select count(*) from dbt_all), 1)"
    " from fct_job_offer_technology t"
    " join dbt_all d on t.job_offer_id = d.job_offer_id"
    " where lower(t.technology) = 'snowflake'"
).df().iloc[0, 0])

dbt_pct = round(100 * dbt_listings / total_listings, 1)

kpi_s = ("background:#1E2128;border:1px solid #2A2E37;border-radius:12px;"
         "padding:20px 24px;flex:1;min-width:140px;text-align:center;")
val_s = ("font-family:'JetBrains Mono',monospace;font-size:28px;"
         "color:#4C8DFF;font-weight:700;")
lbl_s = ("font-family:'Inter',sans-serif;font-size:12px;"
         "color:#9BA1AC;margin-top:4px;")

def _kpi(v, l):
    return (f'<div style="{kpi_s}">'
            f'<div style="{val_s}">{v}</div>'
            f'<div style="{lbl_s}">{l}</div></div>')

HTML('<div style="display:flex;gap:16px;flex-wrap:wrap;margin:16px 0;">'
     + _kpi(dbt_listings, 'listings')
     + _kpi(f'{dbt_pct}%', 'of market')
     + _kpi(dbt_text_mentions, 'text mentions')
     + _kpi(f'{dbt_snow_pct}%', 'Snowflake co&#8209;occurrence')
     + '</div>')

In [15]:
dbt_cotechs = con.execute('''
    with dbt_offers as (
        select distinct job_offer_id
        from fct_job_offer_technology
        where lower(technology) = 'dbt'
    )
    select t.technology,
           count(distinct t.job_offer_id) as offer_count,
           round(100.0 * count(distinct t.job_offer_id)
                 / (select count(*) from dbt_offers), 1) as pct
    from fct_job_offer_technology t
    join dbt_offers d on t.job_offer_id = d.job_offer_id
    where lower(t.technology) != 'dbt'
    group by t.technology
    order by offer_count desc
    limit 15
''').df()
fig = horizontal_bar_chart(
    dbt_cotechs.to_dict('records'), 'technology', 'offer_count',
    'dbt co-technologies',
    labels={r['technology']: f"{int(r['offer_count'])}  ({r['pct']}%)"
            for _, r in dbt_cotechs.iterrows()})
fig

In [16]:
dbt_domains = con.execute('''
    with dbt_offers as (
        select distinct job_offer_id
        from fct_job_offer_technology
        where lower(technology) = 'dbt'
    )
    select d.normalized_domain, count(distinct d.job_offer_id) as offer_count
    from fct_job_offer_domain d
    join dbt_offers db on d.job_offer_id = db.job_offer_id
    where d.normalized_domain in (
        select distinct canonical_domain from mapping_domaines)
    group by d.normalized_domain
    order by offer_count desc
''').df()
fig = horizontal_bar_chart(
    dbt_domains.to_dict('records'), 'normalized_domain', 'offer_count',
    'dbt by domain of intervention')
fig

In [17]:
dbt_employers = con.execute('''
    with dbt_offers as (
        select distinct job_offer_id
        from fct_job_offer_technology
        where lower(technology) = 'dbt'
    )
    select o.employer_category, count(distinct o.job_offer_id) as offer_count
    from fct_job_offer o
    join dbt_offers d on o.job_offer_id = d.job_offer_id
    where o.is_canonical_listing
    group by o.employer_category
    order by offer_count desc
''').df()
fig = column_chart(
    dbt_employers.to_dict('records'), 'employer_category', 'offer_count',
    'dbt by employer category')
fig

In [18]:
dbt_gap = con.execute('''
    select o.job_offer_id, o.job_title,
           regexp_extract(lower(o.job_description),
                          '.{0,60}dbt.{0,60}') as context_snippet
    from fct_job_offer o
    left join fct_job_offer_technology t
        on o.job_offer_id = t.job_offer_id
        and lower(t.technology) = 'dbt'
    where o.is_canonical_listing
      and lower(o.job_description) like '%dbt%'
      and t.job_offer_id is null
''').df()
dbt_gap

,job_offer_id,job_title,context_snippet
0,5042246,Senior Data Engineer Snowflake F/H,ures pratiques de développement et d'automatis...
1,5076954,Run Manager Data Platform F/H,"k, delta lakeorchestration : azure data factor..."
2,213BXYF,Ingénieur data (H/F),projets data avec un environnement data modern...
3,6086171,Data Engineer Snowflake F/H h/f,- réaliser les développements sql et dbt dans ...
4,6084059,Senior Data Analyst - Billing & Collections (H/F),lentes compétences sur la stack data moderne :...
5,2914385,Data Analyst F/H h/f,[python/sql/dbt/ alchemy]
6,5340474,Data Analyst Senior Power BI/GCP (H/F),dbt
7,210VNPK,Architecte Data (H/F) F/H (H/F),* dbt


<div style="margin:24px 0;padding:16px 20px 16px 24px;border-left:3px solid #4C8DFF;background:#1E2128;border-radius:0 8px 8px 0;"><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:15px;color:#F2F3F5;font-weight:700;margin-bottom:6px;">&#128161; The modern data stack footprint</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:13px;color:#9BA1AC;line-height:1.7;">dbt appears in 4.7% of listings but its ecosystem is highly specific: 42% of dbt offers also request Snowflake, 31% Airflow &mdash; forming a clear &ldquo;Modern Data Stack&rdquo; pattern. Direct employers adopt dbt at twice the rate of intermediaries, suggesting the tool spreads from in-house teams outward. The 8 offers where &ldquo;dbt&rdquo; appears in the description but was not extracted as a technology illustrate the extraction model&rsquo;s recall limit on tool names embedded in dense listings.</div></div>

<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">09 &middot; TECH STACK BY ROLE</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Technology penetration by ROME code</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Top 3 ROME codes: M1811 Data Engineer, M1419 Data Analyst, M1405 Data Scientist.</div></div>

In [19]:
tech_M1811 = con.execute('''
    select t.technology, count(distinct t.job_offer_id) as offer_count
    from fct_job_offer o
    join fct_job_offer_technology t on o.job_offer_id = t.job_offer_id
    where o.is_canonical_listing and o.rome_code = 'M1811'
    group by t.technology
    order by offer_count desc
    limit 10
''').df()
fig = horizontal_bar_chart(
    tech_M1811.to_dict('records'), 'technology', 'offer_count',
    'Data Engineer (M1811) — top 10 technologies')
fig

In [20]:
tech_M1419 = con.execute('''
    select t.technology, count(distinct t.job_offer_id) as offer_count
    from fct_job_offer o
    join fct_job_offer_technology t on o.job_offer_id = t.job_offer_id
    where o.is_canonical_listing and o.rome_code = 'M1419'
    group by t.technology
    order by offer_count desc
    limit 10
''').df()
fig = horizontal_bar_chart(
    tech_M1419.to_dict('records'), 'technology', 'offer_count',
    'Data Analyst (M1419) — top 10 technologies')
fig

In [21]:
tech_M1405 = con.execute('''
    select t.technology, count(distinct t.job_offer_id) as offer_count
    from fct_job_offer o
    join fct_job_offer_technology t on o.job_offer_id = t.job_offer_id
    where o.is_canonical_listing and o.rome_code = 'M1405'
    group by t.technology
    order by offer_count desc
    limit 10
''').df()
fig = horizontal_bar_chart(
    tech_M1405.to_dict('records'), 'technology', 'offer_count',
    'Data Scientist (M1405) — top 10 technologies')
fig

<div style="margin:24px 0;padding:16px 20px 16px 24px;border-left:3px solid #4C8DFF;background:#1E2128;border-radius:0 8px 8px 0; box-sizing:border-box;"><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:15px;color:#F2F3F5;font-weight:700;margin-bottom:6px;">&#128161; Three roles, three stacks</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:13px;color:#9BA1AC;line-height:1.7;">Data Engineers own the cloud/infrastructure layer: AWS, Spark, Databricks all rank top-10 for M1811 but are absent from Analyst top-10. Data Analysts live in the BI layer: Power BI ties with SQL at #1 (85 each), and Excel remains in the top 5. Data Scientists stand apart with PyTorch (16 offers), R (14), and Git (22) &mdash; signaling ML production workflows. Python is the only technology that ranks top-3 in <i>all three roles</i>.</div></div>

<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">10 &middot; SKILL CO-OCCURRENCE</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Technology affinity matrix</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Reads: &ldquo;X% of offers requiring row-tech also require column-tech.&rdquo; Diagonal = 100%.</div></div>

In [22]:
# Fetch all (offer, technology) pairs for canonical listings
_pairs = con.execute('''
    select t.job_offer_id, t.technology
    from fct_job_offer_technology t
    join fct_job_offer o on t.job_offer_id = o.job_offer_id
    where o.is_canonical_listing
''').df()

# Top 10 technologies
_top10 = (_pairs.groupby('technology')['job_offer_id']
          .nunique().nlargest(10).index.tolist())

# Binary pivot: offer × technology
_filt = _pairs[_pairs['technology'].isin(_top10)]
_pivot = _filt.pivot_table(
    index='job_offer_id', columns='technology',
    aggfunc='size', fill_value=0)
_pivot = (_pivot > 0).astype(int)

# Co-occurrence matrix: % of A offers that also have B
_techs = [t for t in _top10 if t in _pivot.columns]
_n = len(_techs)
_matrix = np.zeros((_n, _n))
for _i, _a in enumerate(_techs):
    _ca = _pivot[_a].sum()
    for _j, _b in enumerate(_techs):
        if _ca > 0:
            _matrix[_i, _j] = round(100.0 * (_pivot[_a] & _pivot[_b]).sum() / _ca, 1)

fig = go.Figure(go.Heatmap(
    z=_matrix, x=_techs, y=_techs,
    colorscale=[[0, '#15171C'], [0.35, '#1E3A6E'], [0.7, '#3A6EC0'], [1, '#4C8DFF']],
    text=[[f'{v:.0f}%' for v in row] for row in _matrix],
    texttemplate='%{text}',
    textfont=dict(size=9, family=MONO_FONT),
    hovertemplate='%{y} → %{x}: %{text}<extra></extra>',
    showscale=False,
))
fig.update_layout(
    template='dark', height=500, width=700,
    xaxis=dict(side='bottom', tickangle=-45,
               tickfont=dict(family=MONO_FONT, size=10, color=PAPER_MUTED)),
    yaxis=dict(autorange='reversed',
               tickfont=dict(family=MONO_FONT, size=10, color=PAPER_MUTED)),
    margin=dict(l=100, r=24, t=16, b=80))
fig

<div style="margin:24px 0;padding:16px 20px 16px 24px;border-left:3px solid #4C8DFF;background:#1E2128;border-radius:0 8px 8px 0; box-sizing:border-box;"><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:15px;color:#F2F3F5;font-weight:700;margin-bottom:6px;">&#128161; Technologies travel in packs</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:13px;color:#9BA1AC;line-height:1.7;">The matrix reveals distinct ecosystems. Power BI and Tableau show moderate overlap &mdash; they coexist more often than they substitute. Snowflake and Databricks both co-occur heavily with SQL and Python, but their mutual overlap reveals competing cloud-warehouse strategies. AWS and Azure rarely appear together, reflecting cloud vendor lock-in.</div></div>

<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">11 &middot; TECH TENSION</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Technologies associated with the oldest open offers</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Proxy: median age (days since creation_date) of currently active canonical listings, technologies with &ge;10 offers.</div></div>

In [23]:
tension = con.execute('''
    select t.technology,
           median(datediff('day', o.job_offer_creation_date,
                  current_date))::int as median_age_days,
           count(distinct o.job_offer_id) as n
    from fct_job_offer o
    join fct_job_offer_technology t on o.job_offer_id = t.job_offer_id
    where o.is_canonical_listing
    group by t.technology
    having count(distinct o.job_offer_id) >= 10
    order by median_age_days desc
    limit 15
''').df()
fig = horizontal_bar_chart(
    tension.to_dict('records'), 'technology', 'median_age_days',
    'Tech tension — median offer age',
    suffix=' days',
    labels={r['technology']: f"{r['median_age_days']} days (n={r['n']})"
            for _, r in tension.iterrows()},
    color=AMBER)
fig

<div style="margin:24px 0;padding:16px 20px 16px 24px;border-left:3px solid #4C8DFF;background:#1E2128;border-radius:0 8px 8px 0; box-sizing:border-box;"><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:15px;color:#F2F3F5;font-weight:700;margin-bottom:6px;">&#128161; Legacy and niche skills take longer to fill</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:13px;color:#9BA1AC;line-height:1.7;">Qlik and Oracle offers sit open longest, suggesting a shrinking talent pool for legacy BI tools. dbt&rsquo;s high median age is surprising &mdash; it may reflect pickier hiring for modern-stack roles rather than scarcity. Technologies at the bottom of the ranking (mainstream tools like Docker, TensorFlow) fill faster, consistent with a larger available workforce.</div></div>

<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">12 &middot; MARKET DYNAMICS</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Trend, velocity, and flow</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Source: fct_weekly_market (accumulated snapshots) and fct_weekly_market_flow (presence-based flow).</div></div>

In [24]:
weekly = con.execute('''
    select week_start_date, total_offer_count, anonymous_rate_pct,
           direct_employer_rate_pct, median_annual_salary
    from fct_weekly_market
    order by week_start_date
''').df()
fig = line_chart(
    weekly.to_dict('records'), 'week_start_date', 'total_offer_count',
    'Total active offers per week')
fig

In [25]:
flow = con.execute('''
    select week_start_date, weeks_since_previous,
           active_offer_count, new_offer_count,
           exit_count, exit_rate_pct
    from fct_weekly_market_flow
    order by week_start_date
''').df()
flow

,week_start_date,weeks_since_previous,active_offer_count,new_offer_count,exit_count,exit_rate_pct
0,2026-07-13,<NA>,552,552,<NA>,NaN
1,2026-08-24,6,497,408,463,83.9
2,2026-08-31,1,654,179,25,5.0


<div style="margin:24px 0;padding:16px 20px 16px 24px;border-left:3px solid #4C8DFF;background:#1E2128;border-radius:0 8px 8px 0; box-sizing:border-box; box-sizing:border-box; "><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:15px;color:#F2F3F5;font-weight:700;margin-bottom:6px; box-sizing:border-box;">&#128161; A stable market after the summer catch-up</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:13px;color:#9BA1AC;line-height:1.7;">The volume plateau around 960 active listings suggests a steady-state data job market. The 83.9% exit rate on the 6-week gap is a collection artifact (hiatus between pulls), not real churn. True weekly turnover is ~5%, meaning roughly 1 in 20 offers rotates each week.</div></div><div style="margin:24px 0;padding:16px 20px 16px 24px;border-left:3px solid #FFC24B;background:#1E2128;border-radius:0 8px 8px 0;"><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:15px;color:#FFC24B;font-weight:700;margin-bottom:6px;">&#9888;&#65039; Caveat</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:13px;color:#9BA1AC;line-height:1.7;">Seasonality analysis is not feasible with only 4 data points. The creation_date distribution (section 00) shows a ramp from June&ndash;August 2026 but that reflects the corpus build-up, not true seasonal patterns.</div></div>

<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">13 &middot; GEOGRAPHY & TERRITORIES</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Territorial polarization and tech geography</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">IDF = departments 75, 77, 78, 91, 92, 93, 94, 95.</div></div>

In [26]:
geo_zone = con.execute('''
    select
        case when substr(commune_key, 1, 2) = '75'
              or substr(commune_key, 1, 2) = '92'
              or substr(commune_key, 1, 2) = '93'
              or substr(commune_key, 1, 2) = '94'
              or substr(commune_key, 1, 2) = '77'
              or substr(commune_key, 1, 2) = '78'
              or substr(commune_key, 1, 2) = '91'
              or substr(commune_key, 1, 2) = '95'
             then 'Ile-de-France'
             else 'Regions'
        end as zone,
        count(distinct job_offer_id) as offer_count
    from fct_job_offer
    where is_canonical_listing and commune_key is not null
    group by zone
''').df()
total_geo = geo_zone['offer_count'].sum()
geo_zone['label'] = geo_zone.apply(
    lambda r: f"{r['offer_count']}  ({100*r['offer_count']/total_geo:.1f}%)", axis=1)
fig = column_chart(
    geo_zone.to_dict('records'), 'zone', 'offer_count',
    'IDF vs Regions',
    labels=dict(zip(geo_zone['zone'], geo_zone['label'])))
fig

In [27]:
metros = con.execute('''
    select
        case
            when substr(o.commune_key, 1, 2) = '75' then 'Paris'
            when substr(o.commune_key, 1, 2) = '92' then 'Hauts-de-Seine'
            when substr(o.commune_key, 1, 2) = '93' then 'Seine-Saint-Denis'
            when substr(o.commune_key, 1, 2) = '94' then 'Val-de-Marne'
            when substr(o.commune_key, 1, 2) = '69' then 'Lyon metro'
            when substr(o.commune_key, 1, 2) = '31' then 'Toulouse metro'
            when substr(o.commune_key, 1, 2) = '44' then 'Nantes metro'
            when substr(o.commune_key, 1, 2) = '59' then 'Lille metro'
            when substr(o.commune_key, 1, 2) = '33' then 'Bordeaux metro'
            when substr(o.commune_key, 1, 2) = '13' then 'Marseille metro'
            else 'Other departments'
        end as metro,
        count(distinct o.job_offer_id) as offer_count
    from fct_job_offer o
    where o.is_canonical_listing and o.commune_key is not null
    group by metro
    order by offer_count desc
''').df()
fig = horizontal_bar_chart(
    metros.to_dict('records'), 'metro', 'offer_count',
    'Offers by metropolitan area')
fig

In [28]:
tech_geo = con.execute('''
    select t.technology,
        count(distinct case
            when substr(o.commune_key, 1, 2) = '75'
              or substr(o.commune_key, 1, 2) = '92'
              or substr(o.commune_key, 1, 2) = '93'
              or substr(o.commune_key, 1, 2) = '94'
              or substr(o.commune_key, 1, 2) = '77'
              or substr(o.commune_key, 1, 2) = '78'
              or substr(o.commune_key, 1, 2) = '91'
              or substr(o.commune_key, 1, 2) = '95'
            then t.job_offer_id end) as idf,
        count(distinct case
            when not (substr(o.commune_key, 1, 2) = '75'
              or substr(o.commune_key, 1, 2) = '92'
              or substr(o.commune_key, 1, 2) = '93'
              or substr(o.commune_key, 1, 2) = '94'
              or substr(o.commune_key, 1, 2) = '77'
              or substr(o.commune_key, 1, 2) = '78'
              or substr(o.commune_key, 1, 2) = '91'
              or substr(o.commune_key, 1, 2) = '95')
              and o.commune_key is not null
            then t.job_offer_id end) as regions,
        count(distinct t.job_offer_id) as total
    from fct_job_offer_technology t
    join fct_job_offer o on t.job_offer_id = o.job_offer_id
    where o.is_canonical_listing
    group by t.technology
    order by total desc
    limit 15
''').df()

tg_sorted = tech_geo.sort_values('total', ascending=True)
fig = go.Figure()
fig.add_trace(go.Bar(
    y=tg_sorted['technology'], x=tg_sorted['idf'],
    name='Ile-de-France', orientation='h', marker_color=BLUE,
    text=tg_sorted['idf'], textposition='outside',
    textfont=dict(family=MONO_FONT, size=10, color=PAPER)))
fig.add_trace(go.Bar(
    y=tg_sorted['technology'], x=tg_sorted['regions'],
    name='Regions', orientation='h', marker_color=AMBER,
    text=tg_sorted['regions'], textposition='outside',
    textfont=dict(family=MONO_FONT, size=10, color=PAPER)))
fig.update_layout(template='dark', barmode='group',
    height=max(300, 42 * len(tg_sorted) + 60),
    xaxis=dict(showticklabels=False, showgrid=True, gridcolor=HAIRLINE),
    yaxis=dict(showgrid=False),
    margin=dict(l=100, r=60, t=16, b=16))
fig

<div style="margin:24px 0;padding:16px 20px 16px 24px;border-left:3px solid #4C8DFF;background:#1E2128;border-radius:0 8px 8px 0; box-sizing:border-box;"><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:15px;color:#F2F3F5;font-weight:700;margin-bottom:6px;">&#128161; Paris and La D&eacute;fense are the cloud hub</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:13px;color:#9BA1AC;line-height:1.7;">&Icirc;le-de-France captures 47% of data offers, but Hauts-de-Seine (La D&eacute;fense) edges out Paris proper. Cloud and AI tools concentrate in IDF: Dataiku is 3&times; more present in Paris. Talend, by contrast, shows a strong regional skew &mdash; possibly tied to legacy enterprise deployments in manufacturing hubs.</div></div>

<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">14 &middot; EMPLOYERS & CONDITIONS</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Who hires, and on what terms</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Contract types, salary by contract, remote work mentions, and top recruiting employers.</div></div>

In [29]:
top_emps = con.execute('''
    select employer_name, employer_category,
           count(*) as offer_count
    from fct_job_offer
    where is_canonical_listing and employer_name is not null
    group by employer_name, employer_category
    order by offer_count desc
    limit 15
''').df()
top_emps['bar_label'] = top_emps.apply(
    lambda r: f"{r['offer_count']}  ({r['employer_category']})", axis=1)
fig = horizontal_bar_chart(
    top_emps.to_dict('records'), 'employer_name', 'offer_count',
    'Top 15 employers by volume',
    labels=dict(zip(top_emps['employer_name'], top_emps['bar_label'])))
fig

In [30]:
contracts = con.execute('''
    select contract_type,
           count(*) as offer_count,
           round(100.0 * count(*) / (select count(*)
               from fct_job_offer where is_canonical_listing), 1) as pct
    from fct_job_offer
    where is_canonical_listing
    group by contract_type
    order by offer_count desc
''').df()
fig = column_chart(
    contracts.to_dict('records'), 'contract_type', 'offer_count',
    'Contract type breakdown',
    labels={r['contract_type']: f"{r['offer_count']} ({r['pct']}%)"
            for _, r in contracts.iterrows()})
fig

In [31]:
sal_contract = con.execute('''
    select contract_type, count(*) as n,
           median(salary_min) as median_salary
    from fct_job_offer
    where is_canonical_listing
      and salary_period = 'annual' and annual_salary_plausible
    group by contract_type
    order by n desc
''').df()
fig = column_chart(
    sal_contract.to_dict('records'), 'contract_type', 'median_salary',
    'Median salary by contract type', suffix=' €',
    labels={r['contract_type']: f"{int(r['median_salary']):,} € (n={int(r['n'])})".replace(',', ' ')
            for _, r in sal_contract.iterrows()})
fig

In [32]:
remote = con.execute('''
    select employer_category,
           count(distinct job_offer_id) as total,
           count(distinct case when
               lower(job_description) like '%télétravail%'
               or lower(job_description) like '%teletravail%'
               or lower(job_description) like '%remote%'
               or lower(job_description) like '%home office%'
           then job_offer_id end) as remote_count,
           round(100.0 * count(distinct case when
               lower(job_description) like '%télétravail%'
               or lower(job_description) like '%teletravail%'
               or lower(job_description) like '%remote%'
               or lower(job_description) like '%home office%'
           then job_offer_id end)
           / nullif(count(distinct job_offer_id), 0), 1) as remote_pct
    from fct_job_offer
    where is_canonical_listing
    group by employer_category
    order by remote_pct desc
''').df()
fig = column_chart(
    remote.to_dict('records'), 'employer_category', 'remote_pct',
    'Remote work mention rate', suffix='%',
    labels={r['employer_category']: f"{r['remote_pct']}% ({r['remote_count']}/{r['total']})"
            for _, r in remote.iterrows()})
fig

<div style="margin:24px 0;padding:16px 20px 16px 24px;border-left:3px solid #4C8DFF;background:#1E2128;border-radius:0 8px 8px 0; box-sizing:border-box; box-sizing:border-box;"><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:15px;color:#F2F3F5;font-weight:700;margin-bottom:6px; box-sizing:border-box;">&#128161; CDI dominates, but interim pays more</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:13px;color:#9BA1AC;line-height:1.7;">CDI accounts for ~74% of the market. Interim (MIS) shows a ~2.5 k€ salary premium over CDI &mdash; reflecting premium rates for specialized short-term assignments. Only ~20% of offers explicitly mention remote work, with reclassified intermediaries leading at ~41%. This likely under-counts actual remote availability since many employers discuss flexibility at interview stage, not in the posting.</div></div><div style="margin:24px 0;padding:16px 20px 16px 24px;border-left:3px solid #FFC24B;background:#1E2128;border-radius:0 8px 8px 0;"><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:15px;color:#FFC24B;font-weight:700;margin-bottom:6px;">&#9888;&#65039; Caveat</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:13px;color:#9BA1AC;line-height:1.7;">Sector analysis (NAF section from dim_company) is limited to ~16% of listings with a SIREN match. The figures (IT & Telecom ~21%, Consulting ~19%, Manufacturing ~15%) are directional but NOT representative of the full market.</div></div>

In [33]:
con.close()